In [2]:
import os
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import optuna

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.eval.losses import QLIKELoss
from src.eval.scorer import VolatilityScorer
from src.models.causal_transformer_wrapper import CausalTransformerModel
from src.trainer.walk_forward import WalkForwardTrainer
from src.utils import trainer_toolkit as tt
from src.utils import ts_toolkit as tst
from src.utils import test_toolkit as ts

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "causal_transformer"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any()
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any()

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)


## Introducción

El *Causal Transformer* (`CausalTransformerModel`) es una arquitectura de atención secuencial diseñada para capturar dependencias temporales no lineales a lo largo de una ventana histórica de longitud $L$ (`look_back`), garantizando estrictamente que la predicción en cada instante dependa únicamente de observaciones pasadas y presentes. A diferencia de las arquitecturas recurrentes, que propagan el estado oculto de forma secuencial paso a paso, el mecanismo de autoatención calcula relaciones entre todas las posiciones de la ventana en paralelo, permitiendo modelar interacciones entre rezagos distantes sin recurrencia ni dilataciones fijas.

En series financieras, esta formulación permite aprender patrones temporales no lineales entre retornos históricos estandarizados y la varianza futura del crudo Brent, evitando imponer una ecuación paramétrica rígida para la dinámica de la volatilidad condicional.

El objetivo de esta sección es presentar, entrenar y evaluar el modelo `CausalTransformerModel` para pronosticar la varianza realizada de Rogers-Satchell ($RS$) de los futuros del crudo Brent (`BZ=F`) bajo el protocolo del *benchmark*. Se analiza el efecto de la longitud de ventana (`look_back`), la dimensión del espacio latente (`d_model`), el número de cabezas de atención (`nhead`), la profundidad del *encoder* (`num_layers`), el ancho de la red *feed-forward* (`dim_feedforward`), la función de activación y la regularización (`dropout` y `weight_decay`) sobre el desempeño predictivo, manteniendo las mismas condiciones de información, pérdida **QLIKE** y evaluación *walk-forward* que el resto de modelos del proyecto.

### Arquitectura

La arquitectura de `CausalTransformerNetwork` se estructura sobre una proyección lineal de entrada, codificación posicional aprendible, un *encoder* de Transformer con máscara causal superior (`norm_first=True`), normalización final sobre el último paso temporal y una cabeza de salida positiva.

Dado un tensor de entrada $\mathbf{X}_t \in \mathbb{R}^{L \times F}$ (donde $L$ es `look_back` y $F$ es `in_features`, con $F = 1$ para la serie univariada de retornos estandarizados), la proyección lineal y la suma de la codificación posicional aprendible $\mathbf{P} \in \mathbb{R}^{1 \times L \times d_{\text{model}}}$ se expresan como:

$$
\mathbf{H}^{(0)} = \mathbf{X}_t \mathbf{W}_{\text{in}} + \mathbf{b}_{\text{in}} + \mathbf{P},
$$

donde $\mathbf{W}_{\text{in}} \in \mathbb{R}^{F \times d_{\text{model}}}$ y $\mathbf{b}_{\text{in}} \in \mathbb{R}^{d_{\text{model}}}$.

Para evitar cualquier fuga de información hacia adelante (*look-ahead bias*) dentro de la ventana de longitud $L$, se aplica una máscara causal $\mathbf{M} \in \mathbb{R}^{L \times L}$ definida mediante el triángulo superior estricto:

$$
M_{i,j} =
\begin{cases}
0, & \text{si } j \le i, \\
-\infty, & \text{si } j > i.
\end{cases}
$$

En cada capa $l \in \{1, \dots, \text{num\_layers}\}$ del *encoder* con pre-normalización (`norm_first=True`), el mecanismo de atención causal multi-cabeza (*Multi-Head Self-Attention*, MHSA) y el bloque *feed-forward* por posición (FFN) operan residualmente:

$$
\widetilde{\mathbf{H}}^{(l)} = \mathbf{H}^{(l-1)} + \text{MHSA}\left(\text{LayerNorm}\left(\mathbf{H}^{(l-1)}\right), \mathbf{M}\right),
$$

$$
\mathbf{H}^{(l)} = \widetilde{\mathbf{H}}^{(l)} + \text{FFN}\left(\text{LayerNorm}\left(\widetilde{\mathbf{H}}^{(l)}\right)\right),
$$

donde la red *feed-forward* expande la representación hacia `dim_feedforward` (por defecto $2 \cdot d_{\text{model}}$) con *Dropout* y activación seleccionable (`relu` o `gelu` en `_ACTIVATIONS`).

Una vez procesada la secuencia por las capas del *encoder*, se extrae la representación correspondiente al último instante de la ventana, $\mathbf{h}_L^{(\text{num\_layers})} \in \mathbb{R}^{d_{\text{model}}}$, se aplica una capa `LayerNorm` final y se proyecta linealmente a un escalar:

$$
\mathbf{z}_L = \text{LayerNorm}\left(\mathbf{h}_L^{(\text{num\_layers})}\right), \qquad y_t = \mathbf{w}_{\text{fc}}^{\top}\mathbf{z}_L + b_{\text{fc}}.
$$

Para garantizar que la varianza pronosticada sea estrictamente positiva y numéricamente estable, $y_t$ atraviesa la función *Softplus* sumada a un piso $\varepsilon = 10^{-8}$:

$$
\widehat{RS}_t = \text{Softplus}(y_t) + \varepsilon = \ln\left(1 + e^{y_t}\right) + \varepsilon,
$$

donde $\boldsymbol{\theta}$ reúne todos los parámetros entrenables de la proyección inicial, las incrustaciones posicionales, las capas del *encoder*, las normalizaciones y la capa de salida.

Siguiendo el protocolo del *benchmark*, los retornos logarítmicos diarios $r_t$ se estandarizan mediante transformación Z-score ajustada exclusivamente sobre el segmento de entrenamiento de cada *fold* para evitar fuga de información, mientras que la varianza realizada de Rogers-Satchell se escala mediante el factor de objetivo (`target_scale`). El modelo opera bajo un esquema de predicción de un paso adelante (*one-step-ahead forecasting*) y hereda de `BasePyTorchVolatilityModel` la integración con el *walk-forward* (`retrain_freq=1`), optimización con Adam, *weight decay*, *Dropout* y parada temprana (`patience=20`, `max_epochs=200`), minimizando la pérdida asimétrica **QLIKE**:

$$
\boldsymbol{\theta}^{*} = \arg\min_{\boldsymbol{\theta}} \frac{1}{N} \sum_{i=1}^{N} \mathcal{L}_{\text{QLIKE}}\left(RS_{i}, \widehat{RS}_{i}\right),
$$

donde $N$ es el número de ventanas de entrenamiento y $RS_i$ denota la varianza de Rogers-Satchell objetivo. La implementación se desarrolla en `PyTorch` (`2.13.0`) con diferenciación automática.

## Entrenamiento

Para la selección de hiperparámetros del modelo `CausalTransformerModel`, se mantuvo el mismo protocolo experimental (*Walk-Forward*) y el mismo presupuesto computacional que en los modelos paramétricos, recurrentes y convolucionales. A diferencia del GARCH y el EGARCH, la búsqueda de Optuna exploró la configuración de la atención causal y sus parámetros de regularización.

Para adaptar la serie temporal al formato requerido por la red, se construyó una matriz de rezagos (*lag matrix*) para cada ventana del esquema *Walk-Forward* (`n_splits=5`, con `warmup` ajustado según la ejecución y `embargo=1`). Siguiendo los principios de validación cruzada para series temporales (`timeseriescv`), cada fila de esta matriz contenía una secuencia histórica de longitud `look_back`, la cual se utilizó como vector de entrada para predecir el paso inmediatamente posterior. Este enfoque garantizó la integridad temporal de los datos y evitó cualquier tipo de fuga de información futura (*look-ahead bias*) durante el entrenamiento y el reentrenamiento (`retrain_freq=1`).

El espacio de búsqueda incluyó la longitud de la ventana de entrada (`look_back`), la dimensión oculta (`d_model`), el número de cabezas de atención (`nhead`), el número de capas del *encoder* (`num_layers`), la función de activación (`activation`), la tasa de abandono (`dropout`) y la tasa de aprendizaje (`learning_rate`). Para mantener la consistencia dimensional de la red *feed-forward*, `dim_feedforward` se definió como el doble de `d_model`, y para mitigar el sobreajuste se incorporó también la penalización $\ell_2$ (`weight_decay`).

El entrenamiento se configuró con un máximo de 200 épocas y una paciencia de 20 para la parada temprana interna de la red, bajo un esquema de búsqueda con `n_trials=50` y `early_stop_rounds=15`. El desempeño se siguió evaluando mediante la pérdida QLIKE sobre los conjuntos de validación.

In [ ]:
def causal_transformer_search_space(trial: optuna.Trial) -> dict:
    d_model = trial.suggest_categorical("d_model", [32, 64])
    nhead = trial.suggest_categorical("nhead", [2, 4])
    return {
        "look_back": trial.suggest_categorical("look_back", [7, 14, 21, 28]),
        "d_model": d_model,
        "nhead": nhead,
        "num_layers": trial.suggest_int("num_layers", 1, 2),
        "dim_feedforward": d_model * 2,
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "activation": trial.suggest_categorical("activation", ["relu", "gelu"]),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "max_epochs": 200,
        "patience": 20,
        "retrain_freq": 1,
        "seed": 42,
    }


exec_id = 2
current_warmup = 1200 + (exec_id * 10)

exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=CausalTransformerModel,
    scorer=VolatilityScorer(),
    output_dir=exec_dir,
    n_splits=5,
    warmup=current_warmup,
    embargo=1,
    n_trials=50,
    suggest_params_fn=causal_transformer_search_space,
    early_stop_rounds=15,
)

trainer.run(y_train, rs_train)

El algoritmo de búsqueda completó 49 ensayos válidos de 50 posibles (98 %), alcanzando un mejor valor de QLIKE con 0 ensayos podados y 0 fallidos. La ausencia de podas refleja que el codificador con atención causal procesa las ventanas en paralelo y estabiliza su entrenamiento mediante pre-normalización, conexiones residuales y parada temprana, lo que permite completar los ensayos sin descartes prematuros durante la búsqueda.

La búsqueda completa en Optuna tomó aproximadamente 23.50 horas, mientras que el ajuste y entrenamiento del modelo bajo el esquema *Walk-Forward* totalizó aproximadamente 11.41 minutos, manteniendo un bajo tiempo medio por partición. Este tiempo de ajuste evidencia el costo computacional de entrenar arquitecturas basadas en autoatención con Adam y retropropagación frente a modelos más ligeros, conservando la viabilidad operativa dentro del protocolo *Walk-Forward*.

In [4]:
resumen_data = []

exec_id = 2
model_tag = "CausalTransformer" 
study_name_override = None 
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_qlike = "-"

csv_files = sorted(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    if "train_time_sec" in df_csv.columns:
        tiempo_total_rep = float(df_csv["train_time_sec"].sum())
        tiempo_promedio_fold = float(df_csv["train_time_sec"].mean())
    num_folds = len(df_csv)
else:
    print(f"No se encontró el CSV en {exec_dir}")

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        available_studies = optuna.get_all_study_names(storage=storage_url)

        if study_name_override and study_name_override in available_studies:
            target_study = study_name_override
        elif available_studies:
            target_study = available_studies[-1]
        else:
            target_study = f"{model_tag}_study"

        study = optuna.load_study(study_name=target_study, storage=storage_url)

        total_trials = len(study.trials)
        completos = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
        podados = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
        fallidos = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.FAIL)

        if completos > 0 and study.best_value is not None:
            mejor_qlike = round(float(study.best_value), 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")
else:
    print(f"No se encontró la BD en {db_path}")

resumen_data.append({
    "Exec": exec_id,
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Fallidos": fallidos,
    "Mejor QLIKE": mejor_qlike,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3),
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Exec,Total Trials,Completos,Podados,Fallidos,Mejor QLIKE,Tiempo Total (s),Tiempo por Fold (s)
2,50,49,0,0,0.4921,684.7900,2.7720


El ranking de las cinco mejores configuraciones mostró un consenso en el uso de ventanas cortas de 7 días y activación `relu`, lo que sugiere que el historial reciente de una semana resulta informativo para capturar dinámicas de corto plazo en la volatilidad del Brent. En cuanto a la estructura del *encoder*, las configuraciones más destacadas se concentraron en 2 cabezas de atención y profundidades de 1 y 2 capas. Al carecer de un estado oculto recurrente como las redes secuenciales tradicionales, la atención causal sobre la ventana de entrada permite ponderar directamente las relaciones entre pasos temporales dentro del contexto seleccionado sin recurrencia paso a paso.

In [5]:
exec_dir = RESULTS_PATH / "exec_2"

csv_path = exec_dir / "CausalTransformer_results.csv"
if not csv_path.exists():
    fallback_csvs = sorted(exec_dir.glob("*_results.csv"))
    if not fallback_csvs:
        raise FileNotFoundError(f"No se encontró archivo de resultados en {exec_dir}")
    csv_path = fallback_csvs[0]

df_ct = pd.read_csv(csv_path)

df_ct["lr"] = df_ct["learning_rate"].apply(lambda x: f"{x:.4f}")
df_ct["wd"] = df_ct["weight_decay"].apply(lambda x: f"{x:.1e}")
df_ct["drop_fmt"] = df_ct["dropout"].apply(lambda x: f"{x:.3f}")

cols_agrupar = [
    "look_back",
    "d_model",
    "nhead",
    "num_layers",
    "dim_feedforward",
    "drop_fmt",
    "activation",
    "lr",
    "wd",
]

df_agg = (
    df_ct.groupby(cols_agrupar, as_index=False)["qlike"]
    .agg(mean="mean", std="std")
)
df_agg["std"] = df_agg["std"].fillna(0.0)
df_agg["QLIKE"] = df_agg.apply(
    lambda row: f"{row['mean']:.3f} ± {row['std']:.3f}", axis=1
)

df_top5 = df_agg.sort_values(by="mean", ascending=True).head(5).rename(
    columns={
        "look_back": "Lookback",
        "d_model": "d_model",
        "nhead": "Heads",
        "num_layers": "Capas",
        "dim_feedforward": "FFN",
        "drop_fmt": "Dropout",
        "activation": "Act",
        "lr": "LR",
        "wd": "Weight Decay",
    }
)

cols_finales = [
    "Lookback",
    "d_model",
    "Heads",
    "Capas",
    "FFN",
    "Dropout",
    "Act",
    "LR",
    "Weight Decay",
    "QLIKE",
]

df_final = df_top5[cols_finales]
tt.display_styled_df(df=df_final, align_left_first=False, decimals=3)

Lookback,d_model,Heads,Capas,FFN,Dropout,Act,LR,Weight Decay,QLIKE
7,32,2,2,64,0.214,relu,0.0071,6.0e-04,0.492 ± 0.210
7,64,2,1,128,0.246,relu,0.0081,7.4e-04,0.500 ± 0.213
7,64,2,1,128,0.202,relu,0.0077,5.2e-04,0.500 ± 0.243
7,64,2,1,128,0.271,relu,0.0065,5.2e-04,0.506 ± 0.230
7,32,2,2,64,0.097,relu,0.0095,2.4e-04,0.507 ± 0.234


Entre las dos primeras configuraciones, la diferencia de QLIKE es reducida frente a la dispersión observada entre *folds*, lo que hace que ambas sean cercanas en términos de validación. La elección de la primera configuración responde tanto a su menor pérdida media de validación como a un criterio de parsimonia al emplear un espacio latente más compacto (`d_model = 32` y `dim_feedforward = 64`), reduciendo la complejidad y la carga paramétrica del modelo sin sacrificar capacidad predictiva.

In [8]:
import time

scorer = VolatilityScorer()

best_model = CausalTransformerModel(
    look_back=7,
    d_model=32,
    nhead=2,
    num_layers=2,
    dim_feedforward=64,
    dropout=0.214,
    activation="relu",
    learning_rate=0.0071,
    weight_decay=6.0e-4,
    max_epochs=200,
    patience=20,
    batch_size=128,
    retrain_freq=1,
    seed=42,
)

start_time = time.time()
best_model.fit(y_train, rs_train)
end_time = time.time()

fit_time_sec = end_time - start_time
print(f"Entrenamiento completado en {fit_time_sec:.2f} s")

Entrenamiento completado en 35.24 s


In [14]:
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.data.dataset import TimeSeriesDataset
from src.utils import test_toolkit as ts

best_model.net.eval()
look_back = best_model.context_length

y_train_scaled = best_model.scaler.transform(y_train.reshape(-1, 1)).flatten()
rs_train_scaled = rs_train * best_model.target_scale

train_eval_dataset = TimeSeriesDataset(
    y_train_scaled,
    rs_train_scaled,
    look_back,
)
train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=best_model.batch_size,
    shuffle=False,
)

preds_train_list = []
with torch.no_grad():
    for x_batch, _ in train_eval_loader:
        x_batch = x_batch.to(best_model.device)
        preds_batch = best_model.net(x_batch).squeeze(-1).detach().cpu().numpy()
        preds_train_list.append(preds_batch)

preds_train_var = np.concatenate(preds_train_list) / best_model.target_scale
n_preds = len(preds_train_var)

start_idx = len(y_train) - n_preds
dates_aligned = df_train.index[start_idx : start_idx + n_preds]
rs_volatilidad_aligned = np.sqrt(np.maximum(rs_train[start_idx : start_idx + n_preds], 0.0))
pred_volatilidad = np.sqrt(np.maximum(preds_train_var, 0.0))

mask_crisis = (dates_aligned >= "2020-01-01") & (dates_aligned <= "2020-06-30")

ts.plot_volatility_fit(
    dates=dates_aligned,
    rs_variance=rs_volatilidad_aligned,
    pred_variance=pred_volatilidad,
    title="Ajuste dentro de la Muestra de Entrenamiento (Causal Transformer)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad Causal Transformer $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.7),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)

In [2]:
import time

scorer = VolatilityScorer()

best_model = CausalTransformerModel(
    look_back=21,
    d_model=32,
    nhead=4,
    num_layers=2,
    dim_feedforward=64,
    dropout=0.1,
    activation="gelu",
    learning_rate=1e-3,
    weight_decay=1e-4,
    max_epochs=200,
    patience=20,
    batch_size=128,
    retrain_freq=1,
    seed=42,
)

start_time = time.time()
best_model.fit(y_train, rs_train)
end_time = time.time()

elapsed_time = end_time - start_time
print(f">> fit OK en {elapsed_time:.2f} s")
print("   diagnostics:", best_model.get_diagnostics())

n_oos = min(2, len(y_test))
y_test_20 = y_test[-n_oos:]
rs_test_20 = rs_test[-n_oos:]

t0_oos = time.time()
preds_test_20 = best_model.forecast(y_test_20, rs_seq=rs_test_20, debug=False)
oos_time = time.time() - t0_oos

print(
    f">> forecast (últimos {n_oos} días) OK en {oos_time:.2f} s "
    f"| shape={preds_test_20.shape}"
)

>> fit OK en 52.94 s
   diagnostics: {'converged': True, 'epochs_run': 27}
>> forecast (últimos 2 días) OK en 295.66 s | shape=(2,)


In [3]:
import numpy as np
from src.utils import trainer_toolkit as tt

history = best_model.get_history()
diagnostics = best_model.get_diagnostics()

train_loss = np.asarray(history.get("train_loss", []), dtype=float)
val_loss = np.asarray(history.get("val_loss", []), dtype=float)

assert len(train_loss) > 0 and len(val_loss) > 0, "El historial de entrenamiento está vacío."
assert np.isfinite(train_loss).all(), "train_loss contiene NaN o Inf."
assert np.isfinite(val_loss).all(), "val_loss contiene NaN o Inf."

print(
    f"Épocas ejecutadas: {diagnostics.get('epochs_run', len(train_loss))} "
    f"| Convergió: {diagnostics.get('converged')}"
)
print(
    f"Train QLIKE inicial -> final/mínimo: {train_loss[0]:.6f} -> {train_loss[-1]:.6f} "
    f"(min={train_loss.min():.6f})"
)
print(
    f"Val   QLIKE inicial -> final/mínimo: {val_loss[0]:.6f} -> {val_loss[-1]:.6f} "
    f"(min={val_loss.min():.6f})"
)

max_epochs_plotted = len(train_loss)

tt.plot_learning_curve(
    history=history,
    title="Curva de Aprendizaje (Causal Transformer)",
    ylab="Quasi-Likelihood (QLIKE)",
    xlab_pad=14,
    ylab_pad=14,
    line_width=2.0,
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
    ylim=(0, 5),
    xlim=(0, max(10, max_epochs_plotted)),
    y_step=0.5,
    x_step=5,
    legend_bbox=(0.95, 0.95),
)

Épocas ejecutadas: 27 | Convergió: True
Train QLIKE inicial -> final/mínimo: 5.394368 -> 0.369361 (min=0.369361)
Val   QLIKE inicial -> final/mínimo: 3.758891 -> 0.563701 (min=0.465299)


In [6]:
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.data.dataset import TimeSeriesDataset
from src.utils import test_toolkit as ts

best_model.net.eval()
look_back = best_model.context_length

y_train_scaled = best_model.scaler.transform(y_train.reshape(-1, 1)).flatten()
rs_train_scaled = rs_train * best_model.target_scale

train_eval_dataset = TimeSeriesDataset(
    y_train_scaled,
    rs_train_scaled,
    look_back,
)
train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=best_model.batch_size,
    shuffle=False,
)

preds_train_list = []
with torch.no_grad():
    for x_batch, _ in train_eval_loader:
        x_batch = x_batch.to(best_model.device)
        preds_batch = best_model.net(x_batch).squeeze(-1).detach().cpu().numpy()
        preds_train_list.append(preds_batch)

preds_train_var = np.concatenate(preds_train_list) / best_model.target_scale
n_preds = len(preds_train_var)

# Alinear índices según cuántos objetivos construya TimeSeriesDataset
start_idx = len(y_train) - n_preds
dates_aligned = df_train.index[start_idx : start_idx + n_preds]
rs_volatilidad_aligned = np.sqrt(np.maximum(rs_train[start_idx : start_idx + n_preds], 0.0))
pred_volatilidad = np.sqrt(np.maximum(preds_train_var, 0.0))

mask_crisis = (dates_aligned >= "2020-01-01") & (dates_aligned <= "2020-06-30")

ts.plot_volatility_fit(
    dates=dates_aligned,
    rs_variance=rs_volatilidad_aligned,
    pred_variance=pred_volatilidad,
    title="Ajuste dentro de la Muestra de Entrenamiento (Causal Transformer)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad Causal Transformer $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.7),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)